# Современные методы анализа данных и машинного обучения, БИ

## НИУ ВШЭ, 2025-26 учебный год

### Домашнее задание №3. PCA

Задание выполнил(а):

    Бойко Марк

### Общая информация

__Дата выдачи:__ 19.04.2026

__Дедлайн:__ 04:00 12.05.2026

### Оценивание и штрафы

Количество баллов за каждую задачу данного домашнего задания указано рядом с условием задачи.

Оценка за домашнее задание вычисляется по следующей формуле:

$$
s \times 10/70 ,
$$

где $s$  — количество баллов, которое вы набрали в сумме по всем задачам.

За сдачу домашнего задания позже срока на итоговую оценку за задание накладывается штраф в размере 1 **вторичный** балл в день, но  задержка не может быть больше недели.

__Внимание!__ Домашнее задание выполняется самостоятельно. Не допускается помощь в решении домашнего задания от однокурсников или третьих лиц. «Похожие» решения считаются плагиатом, и все задействованные студенты — в том числе и те, у кого списали, — не могут получить за него больше 0 баллов.

Использование в решении домашнего задания генеративных моделей (ChatGPT и так далее) допускается исключительно в рамках справочной и образовательной информации. Любые другие случаи применения средств ИИ — например, для автоматической генерации кода по заданию — считаются плагиатом, и такое домашнее задание оценивается в 0 баллов.

В случае если в решении используются подходы, функции или формулы, не разбиравшиеся на лекциях и семинарах, — необходимо обязательное указание источника: ссылка на документацию, статью, другой ресурс. При отсутствии оформления источников задание может быть полностью обнулено.

### Формат сдачи

Загрузка файлов с решениями происходит в системе [Anytask](https://anytask.org/). Необходимо загружать файл с расширением .ipynb (питоновский ноутбук)

### О задании

В данном задании мы потренируемся в работе с линейной алгеброй и, конкретно, — в применении алгоритма PCA. Цель задания — попробовать сжать звуковую дорожку мелодии Бетховена, используя данный алгоритм снижения размерностей, подробно рассмотренный нами на семинаре и лекции.

Для начала импортируем все необходимые библиотеки:

In [ ]:
from scipy.io import wavfile
from IPython.display import Audio
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

Скачаем данные:

In [ ]:
! wget https://www.dropbox.com/s/p5147nr8mzemxnr/Beethoven_Violin_Sonata_Op_96_first_movement_bars_1-22.wav --quiet

Прочитаем аудиодорожку при помощи wavfile:


In [ ]:
samplerate, data = wavfile.read('Beethoven_Violin_Sonata_Op_96_first_movement_bars_1-22.wav')

В ячейке выше мы с вами видим некую `samplerate`. Данная переменная по умолчанию хранит частоту дискретизации; стандартное для аудио значние — 44100 Гц.

Для тех, кто не очень хорошо знаком со способами кодирования и записи звука (aka тех, кто не сдавал в школе ЕГЭ по информатике :) ), — частота дискретизации показывает, сколько последовательных элементов массива с сигналом кодируют звук длительностью 1 секунда.

Подробнее с кодированием звука можно разобраться [тут](https://ru.wikipedia.org/wiki/Кодирование_звуковой_информации).

Итак, посмотрим, какая же частота дискретизации у нашей аудиодорожки.

In [ ]:
samplerate

Если поделить длину массива сигнала на `samplerate`, получится длительность аудиодорожки в секундах. По крайней мере — должна получиться!


In [ ]:
len(data) / samplerate

Сорок пять секунд с небольшим. Похоже на правду? Сравните с длиной дорожки, запустив её у себя на компьютере.

Заметим также, что звук — стерео, так как сигнал кодируется двумя каналами (для левого и правого динамика).


In [ ]:
data.shape

Отрисуем сигналы в обоих каналах:

In [ ]:
plt.figure(figsize=(20,5))
plt.plot(data[:,0])
plt.show()

plt.figure(figsize=(20,5))
plt.plot(data[:,1])
plt.show()

Вот она — легендарная «кардиограмма» звука, с которой мы все хорошо знакомы! :)

Итак, давайте теперь усредним каналы и получим монозвук, с которым будет существенно проще работать.


In [ ]:
mono_sound = np.mean(data, axis=1)
mono_sound.shape

Ну и теперь мы, наконец, готовы послушать то, что нам собственно предстоит сжимать! :)

In [ ]:
Audio(mono_sound, rate = samplerate)

Прелестно, не правда ли? :)

Для удобства мы также обрежем массив с сигналом таким образом, чтобы его проще было делить на равные части, датасет из которых и необходимо будет сжать известными вам методами.

На самом деле, способ очень похож на тот, которым мы сжимали картинку на семинаре, разбивая её на прямоугольные подкартинки — только здесь задача по большому счету еще проще :)

In [ ]:
mono_sound_to_cut = mono_sound[:1990000]

Проверим, что наш звук теперь — это просто вектор чисел:

In [ ]:
mono_sound_to_cut.shape

Замечательно, всё абсолютно готово!

Ну что же, на этом наша миссия — всё; теперь — ваш выход! :)

### Задание 1


#### 1.1. (2 балла)
Напишите функцию, которая будет разделять сигнал на равные части длиной 1000 и собирать из них "датасет", представляющий собой двумерный массив — матрицу объектов-признаков (каждая часть сигнала длины 1000 находится в отдельной строке).

In [ ]:
# Ваш код здесь

#### 1.2. (3 балла)

Напишите функцию, которая будет переводить вашу "матрицу" обратно в звуковой сигнал. Иными словами, функция разворачивает данные из матрицы размера `(число объектов, 1000)` в вектор длины `(число объектов × 1000)`.

Запустите функцию и проверьте, что все работает корректно, путем воспроизведения "востановленного" сигнала в ноутбуке — этот сигнал должен в точности совпасть с изначальным (ведь на самом деле им и является).

In [ ]:
# Ваш код здесь

#### 1.3. (5 баллов)

Вернемся к матричной форме сонаты. Среди полученных отрывков длины 1000 попробуйте найти похожие отрывки — наиболее "близкие" друг к другу части. Насколько корректно было разделять сонату на отрезки такой длины?

_Подсказка. Для проверки на похожесть можно использовать знакомую вам формулу косинусной близости или любой другой способ._

In [ ]:
# Ваш код здесь

### Задание 2





#### 2.1. (3 балла)

Выполните PCA-преобразование нашей матрицы и получите данные, сжатые в пространство меньшей размерности.

_Подсказка. На этом этапе у нас есть наш "датасет" с 1000 "признаками" и мы хотим уменьшить число "признаков" путем применения метода PCA. Число компонент предлагается выбрать вам, но для начала не стоит брать слишком маленькое число, чтобы потом было проще понять, в случае плохого результата: это компонент оказалось недостаточно или же это вы где-то ошиблись :)_

    

In [ ]:
# Ваш код здесь

#### 2.2. (4 балла)

Визуализируйте наши "объекты" в наглядной форме на плоскости. Сделайте выводы исходя из полученного графика.

_Подсказка. Для этого вам необходимо применить PCA некоторым специальным образом, который мы в том числе обсуждали на семинаре._

In [ ]:
# Ваш код здесь

#### 2.3. (4 балла)

Постройте scatter plot датасета в пространстве, задействовав цвет. Что вообще изображено? Сделайте выводы из этого графика.

_Подсказка. Так мы тоже делали на семинаре. Это еще один другой вариант специального применения PCA._

In [ ]:
# Ваш код здесь

### Задание 3

Нам осталось заняться непосредственно "сжатием" звука и проверкой правильности наших действий.

#### 3.1. (4 балла)

Выполните обратное PCA-преобразование сжатых данных и получите "матрицу" со сжатым звуком. Каков будет размер полученной матрицы?

In [ ]:
# Ваш код здесь

#### 3.2. (4 балла)

Преобразуйте "матрицу", получившуюся в предыдущем пункте, в сигнал, который можно проиграть (наш "сжатый" монозвук). Воспроизведите результат `(Audio(YOUR_RESULT, rate = samplerate)`.

Похож ли полученный звук на оригинал?

In [ ]:
# Ваш код здесь

#### 3.3. (8 баллов)

Проведите исследование зависимости качества звука от числа компонент.

Подберите "на слух" минимальное число компонент, при котором звук практически не отличается от оригинала.

Добавьте в ячейки два варианта звуковой дорожки: оригинальную и выбранную вами. Укажите, какое число компонент вы оставили.

_Подсказка. Попробуйте отфильтровать сигнал с помощью функции `gaussian_filter1d` из `scipy.ndimage`. Это поможет убрать неприятный дробовой шум при сильном сжатии. Вы поймете, о чем идет речь, когда попробуете это на практике._

_Пример кода для фильтрации: `Audio(gaussian_filter1d(mono_sound_compressed, 2), rate = samplerate)`_

In [ ]:
# Ваш код здесь

#### 3.4. (6 баллов)
Аргументированно ответьте на следующие вопросы:
- Количество компонент, которое вы выбрали, это много или мало? Как это можно понять?
- Как сильно можно сжать звук таким образом? Какая оптимизация по памяти у вас получилась в ходе проделанной работы?
- Если нам дадут другую звуковую дорожку, нам нужно будет проделать всё то же самое, чтобы сжать звук. Как автоматически подобрать число компонент, и возможно ли это? Если возможно, то напишите код для автоматического подбора компонент. Если нет, то объясните, почему.




    # Ваш ответ здесь

### Задание 4

#### Дополнительное исследование. (15 баллов)
  
  - Оберните получившийся код по сжатию звука в одну или несколько функций. На вход должен поступать вектор звука, число компонент, а также размер (длительность) одного отрывка. На выходе должен быть сжатый аудиофайл, доступный для воспроизведения.

  - Проведите исследование того, как степень сжатия — соотношение размера частей, на которые делился сигнал в задании 1.1, и размера пространства, в которое вы сжимали данные с помощью PCA — влияет на звук, по субъективным ощущениям. Начиная с какой степени сжатия сильно заметна потеря качества аудиодорожки? (Как с учетом фильтрации с помощью `gaussian_filter1d`, так и без неё). Для сравнения обязательно выведите несколько аудиодорожек при разных степенях сжатия.

  - Что степень сжатия означает для PCA? Для большой аудиозаписи (3 мин, например) мы хотели бы разбить на большее, меньшее или такое же число отрезков, как и для предложенной аудиозаписи? Почему?

  - Можно ли как-то автоматически подобрать степень сжатия? За что она отвечает в нашей задаче? Как степень сжатия влияет на звук? Почему она так влияет на звук? Если степень сжатия можно автоматически подобрать, напишите для этого соответствующий код. Если же нет — объясните почему.



### Задание 5





В этом блоке вы самостоятельно реализуете основные компоненты метода PCA и соберете из них класс, с помощью которого далее осуществите сжатие сонаты для сравнения с классом `PCA` библиотеки `sklearn`.

*Подсказка: для собственной реализации необходимо использовать `SVD`-разложение, которое, напомним, выглядит следующим образом:*

*$$ X = U \cdot \Sigma \cdot V^T $$*

*где U — матрица левых сингулярных векторов, Σ — матрица, на главной диагонали
которой находятся в порядке невозрастания сингулярные числа, транспонированная матрица V — матрица правых сингулярных векторов*

#### 5.1. (3 балла)

Реализуйте метод `fit_transform`. Он принимает на вход первоначальную матрицу и заранее заданное число компонент, находит главные компоненты и возвращает проекцию матрицы на новое число компонент

Вот основные шаги, которые вам необходимо реализовать:

1. Центрирование исходной матрицы
2. `SVD`-разложение (можно использовать модуль `numpy.linalg`)
3. Приведение сингулярных векторов к одному направлению. Подробнее о проблеме неоднозначности знаков можно почитать, например, [тут](https://www.educative.io/blog/sign-ambiguity-in-singular-value-decomposition) (разделы What is the sign ambiguity of SVD? и Ways to handle sign ambiguity). Подумайте, как можно обойти эту проблему в вашей реализации
4. Расчет объясненной дисперсии `explained_variance` (не возвращается функцией, но потребуется для реализации в классе)
5. Расчет доли объясненной дисперсии `explained_variance_ratio` (не возвращается функцией, но потребуется для реализации в классе)
6. Проекция исходной матрицы на n главных компонент
    

In [ ]:
n_comp = ...

def fit_transform(X, n=n_comp):
    # Ваш код здесь


#### 5.2. (5 баллов)

На основе полученной в предыдущем пункте функции создайте класс `CustomPCA`.  

Вот основные шаги, которые вам необходимо реализовать:

1. В конструктор передать число компонент
2. Добавить метод `fit_transform`
3. Реализовать метод `inverse_transform`. Он возвращает трансформированную матрицу в исходное пространство

*Примечание: помните про неоднозначность разложения, о которой говорилось в предыдущем пункте (5.1)*
    

In [ ]:
class CustomPCA:
    def __init__(self, n_components=None):
        # Ваш код здесь

    def fit_transform(X):
        # Ваш код здесь

    def inverse_transform(self, X_transformed):
        # Ваш код здесь


#### 5.3. (4 балла)

Выполните весь пайплайн PCA-преобразования (от выбора разумного для сжатия числа компонент до воспроизведения результата) с использованием вашего класса `CustomPCA`. Сравните на слух полученный результат с работой `PCA`, реализованного в `sklearn`, при одинаковом количестве компонент

Похожи ли результаты? Объясните это совпадение / несовпадение

In [ ]:
# Ваш код здесь